# 16 — SmallCNN1D: ~12k params, SNV+SG1(41,3)

**Change from old CNN (~600k params, RMSE~50)**: model size only.  
Preprocessing, training setup, and splits are unchanged.  

- Preprocessing: SNV -> SG1(window=41, polyorder=3, deriv=1) [confirmed LB-effective]
- Model: SmallCNN1D, ~12k params (Conv1D 8->16->32, AdaptiveAvgPool, small FC)
- Training: Adam(1e-3), CosineAnnealing(T=100), batch=32, early_stopping(patience=20)
- CV: GroupKFold(5) by species — health check only (CV anti-correlated with LB)
- Target: check if overfitting resolves (RMSE from ~50 back to ~20 range)

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission
from src.preprocessing import snv

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_T = 200.0

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'y range: {y.min():.1f} - {y.max():.1f}%')
print(f'Folds: {len(SPLITS)}')

In [ ]:
# ==== SmallCNN1D ====
class SmallCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 8,  kernel_size=15, padding=7),  nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8, 16, kernel_size=9,  padding=4),  nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),  nn.BatchNorm1d(32), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),
        )
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(32 * 8, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.conv(x.unsqueeze(1))
        return self.fc(h.view(x.size(0), -1)).squeeze(1)

# Verify parameter count
_m = SmallCNN1D()
n_params = sum(p.numel() for p in _m.parameters())
n_trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total params    : {n_params:,}')
print(f'Trainable params: {n_trainable:,}')
assert 5000 < n_trainable < 30000, f'Expected ~12k params, got {n_trainable}'
print('Parameter count OK (target: 1-2万台)')

# ==== Preprocessing (fixed: SNV -> SG1(41,3,1)) ====
def preprocess(R):
    A = R.copy().astype(float)
    # SNV (row-wise standardization)
    A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
    # SG1: wide window, low frequency
    A = savgol_filter(A, window_length=41, polyorder=3, deriv=1, axis=1)
    return A.astype(np.float32)

# Precompute full preprocessed arrays
X_pp   = preprocess(X_raw)
X_pp_te = preprocess(X_test_raw)
print(f'Preprocessed: {X_pp.shape}, range [{X_pp.min():.3f}, {X_pp.max():.3f}]')

In [ ]:
def train_model(Xtr, ytr, Xva, yva, n_epochs=100, batch=32, lr=1e-3,
                patience=20, verbose=False):
    # Fold-internal StandardScaler (fit on train only)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr).astype(np.float32)
    Xva_s = sc.transform(Xva).astype(np.float32)

    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                        batch_size=batch, shuffle=True, drop_last=False)

    model = SmallCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.MSELoss()

    best_val  = float('inf')
    best_state = None
    best_preds = None
    no_improve = 0
    loss_log   = []

    for epoch in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            pred = model(xb)
            loss = crit(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(yb)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(Xva_t)
            val_loss = crit(val_pred, yva_t).item()

        loss_log.append({'epoch': epoch+1,
                         'train_rmse': (ep_loss / len(ytr)) ** 0.5,
                         'val_rmse':   val_loss ** 0.5})

        if val_loss < best_val:
            best_val   = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_preds = val_pred.cpu().numpy()
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch + 1) % 10 == 0:
            print(f'    ep{epoch+1:3d}: train_rmse={loss_log[-1]["train_rmse"]:.2f}'
                  f'  val_rmse={loss_log[-1]["val_rmse"]:.2f}')

        if no_improve >= patience:
            if verbose:
                print(f'    Early stop at epoch {epoch+1}')
            break

    # Restore best
    model.load_state_dict(best_state)
    return model, sc, best_preds, loss_log


# Metric helpers
def rmse_all(yt, yp): return float(np.sqrt(np.mean((yt - yp) ** 2)))
def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m] - yp[m]) ** 2))) if m.sum() > 0 else np.nan

print('Training function and metrics defined.')

## GroupKFold CV (health check)

CV metric is for distribution sanity check only, not model selection.  
Goal: confirm overfitting resolved (RMSE dropping from ~50 to ~20 range).

In [ ]:
print('=== GroupKFold CV ===')
print('(CV is health check only — CV anti-correlated with LB)')

fold_results = []
oof_y_list, oof_p_list = [], []
all_loss_logs = []

for fi, (tr, va) in enumerate(SPLITS):
    Xtr, Xva = X_pp[tr], X_pp[va]
    ytr, yva = y[tr],    y[va]

    model_f, sc_f, best_p, llog = train_model(
        Xtr, ytr, Xva, yva, verbose=True)

    r_all = rmse_all(yva, best_p)
    r_le  = rmse_le(yva,  best_p)
    stop_ep = llog[-1]['epoch']

    fold_results.append({'fold': fi+1, 'n_tr': len(tr), 'n_va': len(va),
                         'RMSE_all': round(r_all, 2),
                         'RMSE_le170': round(r_le, 2),
                         'stop_ep': stop_ep})
    oof_y_list.append(yva)
    oof_p_list.append(best_p)
    all_loss_logs.append(llog)

    print(f'  Fold {fi+1}: RMSE_all={r_all:.2f}  RMSE_le170={r_le:.2f}'
          f'  stopped_ep={stop_ep}')

oof_y = np.concatenate(oof_y_list)
oof_p = np.concatenate(oof_p_list)

mean_all = np.mean([r['RMSE_all']   for r in fold_results])
mean_le  = np.mean([r['RMSE_le170'] for r in fold_results])

print()
print(f'  CV mean RMSE_all   : {mean_all:.2f}%')
print(f'  CV mean RMSE_le170 : {mean_le:.2f}%')
print()
print('OOF distribution check:')
print(f'  min={oof_p.min():.1f}  mean={oof_p.mean():.1f}  '
      f'max={oof_p.max():.1f}  std={oof_p.std():.1f}')
print(f'  negative preds: {(oof_p < 0).sum()}')
print(f'  >170%:          {(oof_p > 170).sum()}')

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves per fold
ax = axes[0]
for fi, llog in enumerate(all_loss_logs):
    eps  = [r['epoch']    for r in llog]
    vals = [r['val_rmse'] for r in llog]
    ax.plot(eps, vals, label=f'Fold {fi+1}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val RMSE')
ax.set_title('Validation RMSE per fold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# OOF scatter (y <= 170)
ax = axes[1]
mask = oof_y <= 170
ax.scatter(oof_y[mask], oof_p[mask], s=6, alpha=0.4)
lim = max(oof_y[mask].max(), oof_p[mask].max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=0.8)
ax.set_xlabel('Actual (%)')
ax.set_ylabel('Predicted (%)')
ax.set_title(f'OOF scatter (y<=170, RMSE_le170={mean_le:.2f}%)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/nb16_smallcnn_cv.png', dpi=110)
plt.close()
print('Saved: results/nb16_smallcnn_cv.png')

## Full Train → Test Prediction

In [ ]:
print('=== Training on full train set ===')

# Use average early-stop epoch across folds as training budget
avg_stop = int(np.mean([r['stop_ep'] for r in fold_results]))
print(f'CV avg early-stop epoch: {avg_stop}')

sc_full2 = StandardScaler()
Xtr_full_s = sc_full2.fit_transform(X_pp).astype(np.float32)
Xte_s      = sc_full2.transform(X_pp_te).astype(np.float32)

print(f'Training for {avg_stop} epochs on full train...')
model_final = SmallCNN1D().to(DEVICE)
opt_f  = torch.optim.Adam(model_final.parameters(), lr=1e-3)
sched_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
crit_f  = nn.MSELoss()

Xtr_t = torch.from_numpy(Xtr_full_s).to(DEVICE)
ytr_t = torch.from_numpy(y.astype(np.float32)).to(DEVICE)
loader_f = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)

for ep in range(avg_stop):
    model_final.train()
    for xb, yb in loader_f:
        pred = model_final(xb)
        loss = crit_f(pred, yb)
        opt_f.zero_grad()
        loss.backward()
        opt_f.step()
    sched_f.step()
    if (ep + 1) % 20 == 0:
        model_final.eval()
        with torch.no_grad():
            tr_rmse = (crit_f(model_final(Xtr_t), ytr_t).item()) ** 0.5
        print(f'  ep{ep+1}: train_rmse={tr_rmse:.2f}')

model_final.eval()
with torch.no_grad():
    Xte_t  = torch.from_numpy(Xte_s).to(DEVICE)
    te_pred = model_final(Xte_t).cpu().numpy()

te_pred_clip = np.clip(te_pred, 0, CLIP_T)
print(f'Test predictions: min={te_pred_clip.min():.1f}  '
      f'mean={te_pred_clip.mean():.1f}  '
      f'max={te_pred_clip.max():.1f}  '
      f'>170: {(te_pred_clip > 170).sum()}')

In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

make_submission(test_meta, te_pred_clip, '../submissions/sub_smallcnn.csv')
print('Saved: submissions/sub_smallcnn.csv')

print()
print('=== Summary ===')
print(f'Model params: {sum(p.numel() for p in SmallCNN1D().parameters()):,}')
print(f'Preprocessing: SNV -> SG1(41,3,1)')
print(f'CV RMSE_all (5-fold mean)   : {mean_all:.2f}%')
print(f'CV RMSE_le170 (5-fold mean) : {mean_le:.2f}%')
print(f'Folds: {[r["RMSE_le170"] for r in fold_results]}')
print(f'Test mean={te_pred_clip.mean():.1f}% (healthy if ~40-50%)')
print()
print('LB calibration (SNV-SG1 series, for reference):')
print('  CV=12.96 -> LB=21.20')
print('  CV=14.20 -> LB=19.87')
print('  CV=16.23 -> LB=18.35 (current best)')
print('Note: formula may not apply to CNN architecture.')
print('Submit sub_smallcnn.csv and compare LB directly.')